# 04 - Parse tabela de assuntos

Objetivo: transformar a tabela de assuntos processuais exportada em `.xls`/HTML em um dicionario reutilizavel de codigos, rotulos e caminhos hierarquicos.

A saida principal e `data/reference/assuntos/processed/assuntos_lookup.parquet`, usada pelo notebook `01_exploracao_stj_metadados.ipynb` para trocar codigos como `03608` por rotulos como `Trafico de Drogas e Condutas Afins`.

## 1. Ambiente

Este notebook deve ser executado preferencialmente no ambiente local do projeto, porque a tabela `.xls` esta versionada no repositorio. Depois, se a EDA for rodada no Colab, copie os arquivos gerados em `data/reference/assuntos/processed/` para o Drive em `MyDrive/Mestrado/2026/llms/data/reference/assuntos/processed/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("/Users/felipeignacio/Projects/datajud_probe")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.assuntos import add_hierarchy_paths, parse_assuntos_files, save_assuntos_lookup

PROJECT_ROOT

## 2. Localizar tabelas de assuntos

Por enquanto ha uma tabela de `Justica Federal 1º Grau`. Se depois forem adicionadas tabelas de outras instancias, basta colocar os arquivos em `data/reference/assuntos/raw/` ou em `notebooks/` com nome contendo `Tabela_Assuntos`.

In [ ]:
INPUT_DIRS = [
    PROJECT_ROOT / "data" / "reference" / "assuntos" / "raw",
    PROJECT_ROOT / "notebooks",
]

subject_files = []
for input_dir in INPUT_DIRS:
    if input_dir.exists():
        subject_files.extend(input_dir.glob("*Tabela_Assuntos*.xls"))

subject_files = sorted(set(subject_files))

print(f"Tabelas encontradas: {len(subject_files)}")
for path in subject_files:
    print("-", path.relative_to(PROJECT_ROOT))

if not subject_files:
    raise FileNotFoundError("Nenhuma tabela de assuntos encontrada.")

## 3. Parsear e normalizar

A tabela parece `.xls`, mas o conteudo e HTML. O parser trata esse HTML diretamente, preservando codigo, codigo pai, glossario, ODS, datas e instancia de origem.

In [ ]:
assuntos_df = parse_assuntos_files(subject_files)
assuntos_df = add_hierarchy_paths(assuntos_df)

print(assuntos_df.shape)
assuntos_df.head(10)

In [ ]:
# Checagem rapida do assunto mais recorrente na EDA inicial.
assuntos_df.loc[
    assuntos_df["codigo"].eq("03608"),
    ["codigo", "assunto", "codigo_pai", "caminho_assuntos", "instancia"],
]

## 4. Salvar lookup

O CSV facilita inspecao manual. O Parquet e preferido para leitura automatica no notebook de EDA.

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "reference" / "assuntos" / "processed"
outputs = save_assuntos_lookup(assuntos_df, OUTPUT_DIR)
outputs

In [ ]:
# Amostra compacta para revisao visual.
assuntos_df[["codigo", "assunto", "codigo_pai", "caminho_assuntos"]].head(20)

## 5. Uso na EDA

Depois de gerar o lookup, rode `01_exploracao_stj_metadados.ipynb`. Ele procura automaticamente por `assuntos_lookup.parquet` no repositorio local e, se estiver no Colab, tambem no Drive em `data/reference/assuntos/processed/`.